In [0]:
from pyspark.sql.functions import col, lit, when
from pyspark.sql.types import *

In [0]:
%run "../Refactoring_Reusable_Functions/Data_Ingestion_Refactoring" $CATALOG="tanmay_projects" $SCHEMA="airbnb_bronze" $VOLUME="airbnb_raw" $CITIES="New York" $GET_DATA_URL="https://insideairbnb.com/get-the-data/" $COUNTRY="USA"

In [0]:
# for i in dbutils.fs.ls(city_volume_path.replace('"','')):
#         city_name = i.path.split('/')[-2]
#         schema_name = i.path.split('/')[-3]
#         table_name = i.path.split('/')[-1].replace('.','_')
#         full_table_name = f"{CATALOG}.{schema_name}.{table_name}"
#         print(full_table_name)

dbfs:/Volumes/tanmay_projects/airbnb_bronze/airbnb_raw/new_york/calendar.csv.gz
calendar_csv_gz
new_york
airbnb_raw
tanmay_projects.airbnb_raw.calendar_csv_gz
dbfs:/Volumes/tanmay_projects/airbnb_bronze/airbnb_raw/new_york/listings.csv
listings_csv
new_york
airbnb_raw
tanmay_projects.airbnb_raw.listings_csv
dbfs:/Volumes/tanmay_projects/airbnb_bronze/airbnb_raw/new_york/listings.csv.gz
listings_csv_gz
new_york
airbnb_raw
tanmay_projects.airbnb_raw.listings_csv_gz
dbfs:/Volumes/tanmay_projects/airbnb_bronze/airbnb_raw/new_york/neighbourhoods.csv
neighbourhoods_csv
new_york
airbnb_raw
tanmay_projects.airbnb_raw.neighbourhoods_csv
dbfs:/Volumes/tanmay_projects/airbnb_bronze/airbnb_raw/new_york/neighbourhoods.geojson
neighbourhoods_geojson
new_york
airbnb_raw
tanmay_projects.airbnb_raw.neighbourhoods_geojson
dbfs:/Volumes/tanmay_projects/airbnb_bronze/airbnb_raw/new_york/reviews.csv
reviews_csv
new_york
airbnb_raw
tanmay_projects.airbnb_raw.reviews_csv
dbfs:/Volumes/tanmay_projects/airbnb_

In [0]:
# try:
#     html = get_page_html()
#     download_city_files(CITIES, html)
# except Exception as e:
#     print(f"Download failed for New York City: {e}")
#     #functions call to have the status as failed for airbnb data 

In [0]:
html = get_page_html()
download_city_files_refactoring(CITIES, html)

In [0]:
# def local_variables(iterator):
#     city_name = iterator.path.split("/")[-2].replace("_", "")
#     schema_name = iterator.path.split("/")[-3].replace("raw", "bronze")
#     table_name = iterator.path.split("/")[-1].replace(".", "_")
#     full_table_name = f"{CATALOG}.{schema_name}.{city_name}_{table_name}"
#     dimensions_group = full_table_name.split(".")[-1].split("_")[1]
#     watermark_column_name = "load_ts"
#     watermark_updated_ts = current_timestamp()
#     execution_ts = current_timestamp()
#     src_table_vw_name = "src_table_vw"

#     return [
#         {
#             "city_name": city_name,
#             "schema_name": schema_name,
#             "table_name": table_name,
#             "full_table_name": full_table_name,
#         },
#         {
#             "object_name": full_table_name,
#             "dimensions_group": dimensions_group,
#             "watermark_column_name": watermark_column_name,
#             "watermark_updated_ts": watermark_updated_ts,
#             "execution_ts": execution_ts,
#             "src_table_vw_name": src_table_vw_name,
#         },
#     ]

In [0]:
for iterator in dbutils.fs.ls(city_volume_path.replace('"','')):
    try:
        output=local_variables(iterator)
        object_variables =output[0]
        metadata_variables =output[1]
        # print(object_variables['city_name'])
        # print(object_variables['schema_name'])
        # print(object_variables['table_name'])
        # print(object_variables['full_table_name'])
        # city_name = i.path.split('/')[-2].replace('_','')
        # schema_name = i.path.split('/')[-3].replace('raw','bronze')
        # table_name = i.path.split('/')[-1].replace('.','_')
        # full_table_name = f"{CATALOG}.{schema_name}.{city_name}_{table_name}"
        # print(full_table_name)
        if 'csv' in iterator.name:
            df=spark.read.format("csv").option("header",True).option("inferSchema",True).option("multiLine", True).load(iterator.path) #reading the data
            df.select("*",lit(COUNTRY).alias("Country"),current_timestamp().alias("load_ts")).write.format("delta").mode("append").option("mergeSchema", "true").saveAsTable(object_variables['full_table_name']) #writing the data
            ingestion_control_refactoring(metadata_variables,None,'SUCCESS')
        else:
            df = spark.read.format("json").option("multiLine", True).load(iterator.path) #reading the data
            df.select("*",lit(COUNTRY).alias("Country"),current_timestamp().alias("load_ts")).write.format("delta").mode("append").option("mergeSchema", "true").saveAsTable(object_variables['full_table_name']) #writing the data
            ingestion_control_refactoring(metadata_variables,None,'SUCCESS')
    except Exception as e:
        error_message = str(e)
        ingestion_control_refactoring(metadata_variables,error_message,'FAIL')
        continue


In [0]:
# def ingestion_control_refactoring(metadata_variables,error_message,status):
    
#     # object_name =full_table
#     # dimensions_group = full_table.split('.')[-1].split('_')[1]
#     # watermark_column_name = 'load_ts'
#     # watermark_updated_ts = current_timestamp()
#     error_message = error_message
#     status = status
#     # execution_ts = current_timestamp()
#     # src_table_vw_name = 'src_table_vw'

#     df = spark.range(1).select(
#     lit(metadata_variables['object_name']).alias('object_name'),
#     lit(metadata_variables['dimensions_group']).alias('dimensions_group'),
#     lit(metadata_variables['watermark_column_name']).alias('watermark_column_name'),
#     metadata_variables['watermark_updated_ts'].alias('watermark_updated_ts'),
#     lit(error_message).alias('error_message'),
#     lit(status).cast("string").alias("status"),
#     metadata_variables['execution_ts'].alias("execution_ts")
#     ).createOrReplaceTempView(metadata_variables['src_table_vw_name'])

#     spark.sql(f"""merge into {ingestion_table_name} IC
#                 using {metadata_variables['src_table_vw_name']} SV
#                 on IC.object_name = SV.object_name and IC.execution_ts = SV.execution_ts
#                 when matched then update set IC.watermark_updated_ts = SV.watermark_updated_ts, IC.status = SV.status, IC.error_message = SV.error_message
#                 when not matched then insert
#                 (object_name, dimensions_group, watermark_column_name, watermark_updated_ts, error_message, status, execution_ts)
#                 values
#                 (SV.object_name, SV.dimensions_group, SV.watermark_column_name, SV.watermark_updated_ts, SV.error_message, SV.status, SV.execution_ts)""")

In [0]:
# # --- Load everything currently in the volume into tables ---
# loaded = []
# for src in list_raw_files():
#     table = load_file_to_table_refactoring(src)
#     if table:
#         rows = spark.table(table).count()
#         loaded.append((src, table, rows))

# if not loaded:
#     raise ValueError(f"No files found under {VOLUME_PATH} — run the ingestion cell first.")

# display(spark.createDataFrame(loaded, ["source_file", "target_table", "row_count"]))

tanmay_projects.airbnb_bronze.new_york_listings_csv
tanmay_projects.airbnb_bronze.new_york_calendar_csv
tanmay_projects.airbnb_bronze.new_york_reviews_csv
tanmay_projects.airbnb_bronze.new_york_listings
tanmay_projects.airbnb_bronze.new_york_reviews
tanmay_projects.airbnb_bronze.new_york_neighbourhoods
tanmay_projects.airbnb_bronze.new_york_neighbourhoods_geojson


source_file,target_table,row_count
/Volumes/tanmay_projects/airbnb_bronze/airbnb_raw/new_york/listings.csv.gz,tanmay_projects.airbnb_bronze.new_york_listings_csv,9500
/Volumes/tanmay_projects/airbnb_bronze/airbnb_raw/new_york/calendar.csv.gz,tanmay_projects.airbnb_bronze.new_york_calendar_csv,2737500
/Volumes/tanmay_projects/airbnb_bronze/airbnb_raw/new_york/reviews.csv.gz,tanmay_projects.airbnb_bronze.new_york_reviews_csv,423514
/Volumes/tanmay_projects/airbnb_bronze/airbnb_raw/new_york/listings.csv,tanmay_projects.airbnb_bronze.new_york_listings,6500
/Volumes/tanmay_projects/airbnb_bronze/airbnb_raw/new_york/reviews.csv,tanmay_projects.airbnb_bronze.new_york_reviews,393263
/Volumes/tanmay_projects/airbnb_bronze/airbnb_raw/new_york/neighbourhoods.csv,tanmay_projects.airbnb_bronze.new_york_neighbourhoods,208
/Volumes/tanmay_projects/airbnb_bronze/airbnb_raw/new_york/neighbourhoods.geojson,tanmay_projects.airbnb_bronze.new_york_neighbourhoods_geojson,13


In [0]:
%sql
#need to check dimension group
select * from tanmay_projects.airbnb_bronze.ingestion_control

object_id,object_name,dimensions_group,watermark_column_name,watermark_updated_ts,error_message,status,execution_ts
26,tanmay_projects.airbnb_bronze.new_york_neighbourhoods_geojson,neighbourhoods,load_ts,2026-11-16T18:22:28.097Z,null,null,2026-11-16T18:22:28.097Z
21,tanmay_projects.airbnb_bronze.new_york_calendar_csv,calendar,load_ts,2026-11-16T18:21:54.856Z,null,null,2026-11-16T18:21:54.856Z
20,tanmay_projects.airbnb_bronze.new_york_listings_csv,listings,load_ts,2026-11-16T18:21:47.352Z,null,null,2026-11-16T18:21:47.352Z
22,tanmay_projects.airbnb_bronze.new_york_reviews_csv,reviews,load_ts,2026-11-16T18:22:01.561Z,null,null,2026-11-16T18:22:01.561Z
25,tanmay_projects.airbnb_bronze.new_york_neighbourhoods,york,load_ts,2026-11-16T18:22:21.574Z,null,null,2026-11-16T18:22:21.574Z
23,tanmay_projects.airbnb_bronze.new_york_listings,york,load_ts,2026-11-16T18:22:08.363Z,null,null,2026-11-16T18:22:08.363Z
24,tanmay_projects.airbnb_bronze.new_york_reviews,york,load_ts,2026-11-16T18:22:14.790Z,null,null,2026-11-16T18:22:14.790Z
16,tanmay_projects.airbnb_bronze.new_york_neighbourhoods_geojson,york,load_ts,null,null,null,null
9,tanmay_projects.airbnb_bronze.new_york_neighbourhoods_geojson,york,load_ts,null,null,null,null
8,tanmay_projects.airbnb_bronze.new_york_neighbourhoods,york,load_ts,null,null,null,null
